# Week 04：AI 應用 + Python（下）

## AI互動與API基礎操作・用Gemini API擴充聊天機器人

> 🤖 **本課程的核心工作流：你出主意，AI 寫程式，你來驗證**
>
> 💡 每個範例都附有 Prompt，你可以直接複製貼到 ChatGPT / Gemini 試試看！

---

## 🕙 Round 1：直播教學（10:10 - 10:40）

### 🎯 本堂課學習目標

- **LLM基礎概念** - AI大腦是怎麼運作的 🤖
- **API是什麼** - 用程式呼叫AI的「電話」 ☎️
- **申請Gemini API Key** - 拿到你的專屬鑰匙 🔑
- **第一次對話** - 讓模型回答問題 💬
- **擴充功能** - 連續對話、角色代入、串流輸出 🚀

> ⚠️ **開始前準備**：
> 1. 到 [ai.google.dev/gemini-api](https://ai.google.dev/gemini-api/docs?hl=zh-tw) 申請你的 API Key（免費！）
> 2. 🔑 Key 就像你家門的鑰匙，**千萬不要給別人看、不要上傳到公開的地方！**

---

## 1️⃣ 準備：安裝套件 + 設定你的 API Key 🔑

> 💬 **Prompt**：「我要在 Colab 用 Python 呼叫 Gemini，要先裝什麼套件？API Key 要怎麼設定？」

> ⚠️ 把下面 `GEMINI_API` 的空白處換成你自己的 Key！

In [ ]:
# 安裝所需的套件（新版 SDK：google-genai）
!pip install google-genai

In [ ]:

from google import genai   # 新版 SDK（取代舊的 google-generativeai）

GEMINI_API = "YOUR_API_KEY_HERE"   # 🔑 把你的 API Key 貼在這裡！
MODEL_NAME = "gemma-4-26b-a4b-it"

client = genai.Client(api_key=GEMINI_API)
print("✅ 準備完成，可以開始跟 AI 對話了！")

---

## 2️⃣ 🚀 第一次跟模型對話！

> 💬 **Prompt**：「幫我寫一個Python程式，用google-genai呼叫Gemini，問它『請用繁體中文介紹AI是如何運作的』」

> 🎉 按 `Shift + Enter` 執行，你就成功跟AI對話了！
>
> 🧠 **小知識**：有些模型會「先思考、再回答」。程式裡已自動跳過它的思考過程（`thought=True` 的片段），只顯示最終答案。

In [ ]:
response = client.models.generate_content(
    model=MODEL_NAME,
    contents="請用繁體中文介紹AI是如何運作的"
)
# print(response.text)
# 有些模型會「先思考再回答」：thought=True 的片段是它的內心獨白（思考過程）
# 我們只顯示最終答案，跳過思考部分
for part in response.candidates[0].content.parts:
    if not part.thought:
        print(part.text or "", end="")
print()

# 💡 換成任何問題試試看：「為什麼天空是藍色的？」

In [ ]:
response = client.models.generate_content(
    model=MODEL_NAME,
    contents="為什麼天空是藍色的？"
)

for part in response.candidates[0].content.parts:
    if not part.thought:
        print(part.text or "", end="")
print()


---

## 3️⃣ 連續對話（while 迴圈）🔄

> 💬 **Prompt**：「幫我寫一個無限迴圈的對話程式，輸入空白就結束」

> 🤔 **想一想**：這樣連續問，AI 記得上一句嗎？試試問它「我剛剛說了什麼？」

In [ ]:
# 連續對話
while True:
    in_text = input('輸入:')
    if in_text == '': break
    response = client.models.generate_content(model=MODEL_NAME, contents=in_text)
    print(response.text)
print("對話結束！")

---

## 4️⃣ 代入角色對話 🎭

> 💬 **Prompt**：「讓Gemini扮演『寶可夢訓練家小智』來回答問題」

> 💡 把「角色設定」寫進提示詞（prompt），AI 就會入戲！這招叫 **Prompt Engineering** 🪄
> 換成你喜歡的歷史人物、動漫角色試試看！

In [ ]:
prompt = """你現在是《寶可夢》的主角小智，
請用小智的語氣回答我的問題。

我的問題：誰是最強的寶可夢？"""

response = client.models.generate_content(model=MODEL_NAME, contents=prompt)
print(response.text)

---

## 5️⃣ 以串流方式輸出字 📺

- 不用等全部生成完才顯示，可以**一個字一個字蹦出來**
- 就像看直播打字機效果，等待感瞬間消失！✨

> 💬 **Prompt**：「讓Gemini的回答一個字一個字顯示出來（串流），要怎麼寫？」

In [ ]:
# 新版 SDK：串流有專屬方法 generate_content_stream()
response = client.models.generate_content_stream(
    model=MODEL_NAME,
    contents="請用三句話形容程式設計"
)
for chunk in response:
    # print(chunk.text, end="")
    # 一樣跳過「思考」片段（thought=True），只顯示最終答案
    for part in chunk.candidates[0].content.parts:
        if not part.thought:
            print(part.text or "", end="")
print()  # 換行

---

## 6️⃣ 調整參數 🎛️

| 參數 | 作用 | 比喻 |
|------|------|------|
| `temperature` | 控制回答的「隨機性」 | 0 = 穩重老實，1+ = 創意爆棚（也可能胡說）🎲 |
| `max_tokens` | 回答的最長長度 | 限制作文字數上限 📏 |

> 🎮 **小實驗**：同一個問題，用 temperature=0 和 temperature=1.5 各問一次，看看回答差多少！

In [ ]:
question = "幫我的寵物貓取一個有創意的名字"

# 穩重模式：temperature = 0（config 可以直接傳字典，最簡單！）
print("【穩重模式】")
response = client.models.generate_content(
    model=MODEL_NAME,
    contents=question,
    config={"temperature": 0}
)
print(response.text)

# 瘋狂模式：temperature = 1.5
print("\n【瘋狂模式】")
response = client.models.generate_content(
    model=MODEL_NAME,
    contents=question,
    config={"temperature": 1.5}
)
print(response.text)

---

## ✏️ 自由練習區（Round 1：10:40 - 11:00）

### 練習任務：
1. 🔑 申請你的 Gemini API Key（還沒拿到的同學趕快！）
2. 💬 在 Colab 跑通「第一次跟模型對話」的程式碼
3. 🎭 用角色代入問 AI 三個你好奇的問題（例如讓它扮演歷史人物、動漫角色……）

> 💡 **提示**：卡住了可以問 AI！但要先自己試看看～
> 覺得太簡單？試試把 temperature 調到 1.5，看它會怎麼「發瘋」😆

In [ ]:
# ✏️ 練習區：把你的程式碼寫在這裡！
# 提示：可以複製上面的角色代入範例，換成你喜歡的角色和問題

---

## 🕐 Round 2：擴充聊天機器人功能（11:10 - 11:30）

### 🤔 上堂課的機器人有個大問題……

- 它**沒有記憶**！每句話都是第一次見面 😅
- 「我剛剛問了你什麼？」→「？？？」

> 今天來解決：**讓 AI 記住對話！** 🧠

---

## 7️⃣ 方法一：用 `client.chats.create()`（最簡單）✅

> 💬 **Prompt**：「讓Gemini的對話程式記得之前的訊息，要怎麼寫？」

> 💡 `chats.create()` 會自動把**所有歷史訊息**一起帶給模型，AI 就記得上下文了！
> 試試問它：「我上一句說了什麼？」

In [ ]:
chat = client.chats.create(model=MODEL_NAME)   # 建立一個有記憶的對話

while True:
    in_text = input('輸入:')
    if in_text == '': break
    response = chat.send_message(in_text)
    print(response.text)
print("對話結束！")

---

## 8️⃣ 方法二：傻瓜法（手動拼歷史）📜

- 自己用 list 記錄每一句話，每次查詢前把歷史**全部拼起來**送進去

> ⚠️ 缺點：聊得越久，每次送的字越多 → 變慢、變貴（長到超過模型上限就裝不下了）

In [ ]:
messages = [
    {"role": "user", "content": "誰是小智？"},
    {"role": "assistant", "content": "小智是《寶可夢》動畫的主角！"},
]

# 拼接對話歷史為上下文
query_context = "\n".join(f"{m['role']}: {m['content']}" for m in messages)
print(query_context)

# 新增一個訊息，詢問「我的上一個問題是什麼？」
new_question = "我的上一個問題是什麼？"
new_message = {"role": "user", "content": new_question}
print(f"新訊息: {new_message['role']}: {new_message['content']}")

# 傻瓜法：把「歷史 + 新訊息」拼成一大段 prompt，一次送進去（土炮版記憶功能）
full_prompt = query_context + "\n" + f"{new_message['role']}: {new_message['content']}"
response = client.models.generate_content(model=MODEL_NAME, contents=full_prompt)
print(response.text)


---

## 9️⃣ 另一種套件：llama_index 📦（選做）

- 除了 `google-genai`，還有其他套件也能呼叫 Gemini
- llama_index 功能更多（之後的 RAG 課程會用到！）

> 💬 **Prompt**：「用 llama_index 呼叫 Gemini 要怎麼安裝和使用？」

> 🧩 就像同一件事可以用不同品牌的工具完成，**學會一種，其他都類推！**

In [ ]:
# 安裝 LlamaIndex 套件（支援 Gemini LLM）
!pip install llama-index
!pip install llama-index-llms-gemini

In [ ]:
from llama_index.llms.gemini import Gemini

response = Gemini(api_key=GEMINI_API, model=MODEL_NAME).complete("誰是小智？")
print(response)

---

## 🔟 不用 SDK：直接發 HTTP 請求（URL API）🔗

- SDK 只是「包裝紙」！底層其實都是發 **HTTP 請求**到一個固定網址（endpoint）
- Gemini REST endpoint 格式：`POST /v1beta/models/{model}:generateContent`
- 版本號前綴 `/v1beta/` + 動作後綴 `:generateContent`，這就是「通用 API URL 格式」

> 💬 **Prompt**：「不用任何套件，用 requests 直接呼叫 Gemini REST API 要怎麼寫？」

In [ ]:
import requests

# 通用 API URL：版本號前綴 /v1beta/ + 模型名 + 動作後綴 :generateContent
url = f"https://generativelanguage.googleapis.com/v1beta/models/{MODEL_NAME}:generateContent"

headers = {"x-goog-api-key": GEMINI_API}   # 🔑 API Key 放在請求標頭裡驗證身分

payload = {
    "contents": [
        {"parts": [{"text": "誰是小智？"}]}   # JSON 格式：送訊息給模型
    ]
}

response = requests.post(url, headers=headers, json=payload)
print("HTTP 狀態碼:", response.status_code)   # 200 = 成功

# 回傳也是 JSON，從裡面把文字挖出來
# answer = response.json()["candidates"][0]["content"]["parts"][0]["text"]

# 回傳也是 JSON，從裡面把文字挖出來（一樣跳過 thought=True 的思考片段）
parts = response.json()["candidates"][0]["content"]["parts"]
answer = "".join(p["text"] for p in parts if not p.get("thought"))
print(answer)

---

## 📝 本堂課作業（下次上課前繳交）

### 🤖 打造你的「有記憶」AI聊天機器人！

1. ☁️ 寫一個**有連續對話功能**的聊天程式（用 `client.chats.create()`）
2. 🎭 加入**角色代入**：讓 AI 扮演你喜歡的角色（動漫人物、歷史名人、甚至你自己……）
3. 📺 加上**串流輸出**，讓回答像打字機一樣蹦出來
4. 🎲 **進階挑戰（選做）**：用 `temperature` 做出「認真模式 / 搞笑模式」切換

> 💡 **提示**：請 AI 幫忙完全 OK！但你要能解釋程式在幹嘛，角色設定要你自己想～

In [ ]:
# 📝 作業區：把你的「有記憶」聊天機器人寫在這裡！
# 提示：結合 client.chats.create() + 角色代入 prompt + 串流輸出